# Wine Quality — Exploratory Data Analysis

Supporting analysis for **PMLDL Assignment 1**. It documents the decisions that are
implemented in the automated pipeline:

* why the target is binarised at `quality >= 6`,
* why duplicates are dropped,
* why outliers are removed with a **3·IQR** rule,
* which engineered features were added in `code/models/features.py`.

> The notebook is **documentation only** — the pipeline itself never runs it.
> Run it with `pip install -r requirements.txt matplotlib seaborn jupyter`.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / 'code'))
sys.path.insert(0, str(PROJECT_ROOT / 'code' / 'datasets'))
sys.path.insert(0, str(PROJECT_ROOT / 'code' / 'models'))

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 40)


## 1. Load the raw data

Exactly the same loading routine that stage 1 of the pipeline uses.


In [ ]:
from prepare_data import load_raw_data

raw = load_raw_data()
print(raw.shape)
raw.head()


In [ ]:
raw.describe().T


In [ ]:
print('missing values per column:')
print(raw.isna().sum())
print()
print('exact duplicate rows:', raw.duplicated().sum())
print('wine types:')
print(raw['wine_type'].value_counts())


The public dataset contains **no missing values**, but it does contain ~1 177 exact
duplicates. The pipeline still implements median imputation per wine type so that it keeps
working if a future data drop is incomplete (this behaviour is covered by
`tests/test_data_engineering.py`).


## 2. The target: why `quality >= 6`?


In [ ]:
ax = raw['quality'].value_counts().sort_index().plot(kind='bar', figsize=(7, 3.5))
ax.set_title('Distribution of the expert quality score')
ax.set_xlabel('quality')
ax.set_ylabel('number of wines')
plt.tight_layout()
plt.show()

binary = (raw['quality'] >= 6).astype(int)
print('class balance for `quality >= 6`:')
print(binary.value_counts(normalize=True).round(3))


Scores 3–9 are heavily concentrated on 5 and 6, so a 7-class problem is mostly noise.
Splitting at 6 gives a **~63 / 37 split** — imbalanced enough to be interesting, balanced
enough to train on directly (the pipeline additionally uses class-weighted estimators).


## 3. Outliers


In [ ]:
features = [c for c in raw.select_dtypes('number').columns if c != 'quality']

q1 = raw[features].quantile(0.25)
q3 = raw[features].quantile(0.75)
iqr = q3 - q1

for multiplier in (1.5, 3.0):
    mask = ((raw[features] < (q1 - multiplier * iqr)) | (raw[features] > (q3 + multiplier * iqr))).any(axis=1)
    print(f'{multiplier:>4} x IQR -> {mask.sum():5d} rows removed ({mask.mean():.1%})')


In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(15, 8))
for ax, column in zip(axes.ravel(), features):
    raw.boxplot(column=column, by='wine_type', ax=ax)
    ax.set_title(column)
    ax.set_xlabel('')
fig.suptitle('')
plt.tight_layout()
plt.show()


The classic 1.5·IQR fence would delete ~20 % of the data — far too aggressive for a
dataset whose tails are genuine wine chemistry. The pipeline therefore uses **3·IQR**
(≈5 % of the rows) and caps the removal at 15 % (`params.yaml → data.max_outlier_fraction`).


## 4. Which features carry the signal?


In [ ]:
corr = raw[features + ['quality']].corr()['quality'].drop('quality').sort_values()
ax = corr.plot(kind='barh', figsize=(7, 5))
ax.set_title('Pearson correlation with the quality score')
plt.tight_layout()
plt.show()
corr.round(3)


`alcohol` is by far the strongest single predictor, followed by `density` and
`volatile_acidity` (vinegary taste) — which motivates the engineered ratios below.


## 5. Engineered features

The exact transformer that is embedded in the deployed model artifact.


In [ ]:
from features import ENGINEERED_FEATURES, add_engineered_features

enriched = add_engineered_features(raw)
enriched[ENGINEERED_FEATURES].describe().T


In [ ]:
target = (raw['quality'] >= 6).astype(int)
signal = enriched[ENGINEERED_FEATURES].corrwith(target).sort_values()
ax = signal.plot(kind='barh', figsize=(7, 3.5))
ax.set_title('Correlation of the engineered features with `is_good_quality`')
plt.tight_layout()
plt.show()
signal.round(3)


## 6. Conclusions fed back into the pipeline

| Observation | Implementation |
| --- | --- |
| 1 177 exact duplicates | dropped in `prepare_data.clean_data` |
| no missing values today, but the loader must be robust | median-per-wine-type imputation |
| 1.5·IQR is too aggressive, 3·IQR is reasonable | `params.yaml → data.iqr_multiplier: 3.0` |
| the quality score is concentrated on 5–6 | binary target `quality >= 6` |
| alcohol/density/volatile acidity dominate | ratio features in `code/models/features.py` |
| classes are imbalanced (~63/37) | class-weighted candidates + **F1** as the selection metric |
